# 基于宏观因子的大类资产配置框架
## 国泰金工研报复现 - 大类资产配置量化模型研究系列之四

本notebook复现国泰金工研报《基于宏观因子的大类资产配置框架》的核心内容。

### 主要内容
1. **六大宏观因子体系**: 增长、通胀、利率、信用、汇率、流动性
2. **四步配置框架**:
   - 选取合适的因子
   - 计算资产的因子暴露
   - 确定因子目标暴露
   - 匹配因子目标暴露
3. **因子偏离实证分析**

In [ ]:
# 添加source目录到系统路径
import sys
sys.path.insert(0, '../source')

import warnings
warnings.filterwarnings('ignore')

print('环境配置完成')

## 1. 数据准备

In [ ]:
from source.config import ALL_ASSETS, MACRO_FACTORS, BACKTEST_CONFIG, OUTPUT_DIR
from source.data_fetcher import load_or_fetch_data, generate_simulated_data

# 设置回测参数
start_date = BACKTEST_CONFIG['start_date']
end_date = BACKTEST_CONFIG['end_date']
print(f'回测期间: {start_date} 至 {end_date}')
print(f'资产列表: {ALL_ASSETS}')

In [ ]:
# 获取资产收益率数据
# use_simulated_data=True 使用模拟数据（真实数据需要API权限）
asset_returns = generate_simulated_data(start_date, end_date)

print(f'\n资产收益率数据形状: {asset_returns.shape}')
print(f'数据时间范围: {asset_returns.index[0]} 至 {asset_returns.index[-1]}')
print(f'\n资产列表: {asset_returns.columns.tolist()}')

In [ ]:
# 查看数据样本
asset_returns.head(10)

## 2. 构建宏观因子

In [ ]:
from source.macro_factors import MacroFactorBuilder

# 初始化因子构建器
factor_builder = MacroFactorBuilder(n_factors=6)

# 构建六大宏观因子
factor_returns = factor_builder.construct_all_factors(asset_returns)

print(f'因子收益率数据形状: {factor_returns.shape}')
print(f'\n因子列表: {factor_returns.columns.tolist()}')

In [ ]:
# 查看因子数据样本
factor_returns.head(10)

In [ ]:
# 因子统计描述
factor_returns.describe()

In [ ]:
# 因子相关性矩阵
import numpy as np

factor_corr = factor_returns.corr()
print('因子相关性矩阵:')
factor_corr.round(3)

## 3. 计算因子暴露

In [ ]:
from source.factor_exposure import FactorExposureWithPrior

# 初始化因子暴露计算器（带先验信息）
exposure_calculator = FactorExposureWithPrior(alpha=0.01)

# 计算资产对各因子的暴露
exposure_matrix = exposure_calculator.fit(asset_returns, factor_returns)

print(f'因子暴露矩阵形状: {exposure_matrix.shape}')
print(f'\n资产列表: {exposure_matrix.index.tolist()}')
print(f'因子列表: {exposure_matrix.columns.tolist()}')

In [ ]:
# 查看因子暴露矩阵
exposure_matrix.round(4)

In [ ]:
# 计算残差风险
residual_returns = exposure_calculator.predict_asset_returns(factor_returns)

# 实际收益与预测收益对比
common_idx = asset_returns.index.intersection(residual_returns.index)
residuals = asset_returns.loc[common_idx] - residual_returns.loc[common_idx]
heterogeneous_var = residuals.var()

print('各资产异质风险方差:')
heterogeneous_var.round(6)

## 4. 因子偏离回测

In [ ]:
from source.backtest import BacktestEngine, BacktestResultAnalyzer

# 初始化回测引擎
backtest_engine = BacktestEngine(
    start_date=start_date,
    end_date=end_date,
    rebalance_freq='monthly',
    factor_deviation=0.05
)

print('回测引擎初始化完成')
print(f'调仓频率: {backtest_engine.rebalance_freq}')
print(f'因子偏离值: {backtest_engine.factor_deviation}')

In [ ]:
# 对每个因子进行偏离回测
print('开始运行因子偏离回测...')
print('=' * 50)

all_results = {}

for factor in MACRO_FACTORS:
    print(f'\n回测 {factor} 因子偏离策略...')
    result = backtest_engine.run_factor_deviation_backtest(
        asset_returns, factor_returns, target_factor=factor
    )
    all_results[factor] = result
    
    portfolio_values = result['portfolio_values']
    if len(portfolio_values) > 1:
        total_return = (portfolio_values.iloc[-1] / portfolio_values.iloc[0] - 1) * 100
        print(f'  总收益率: {total_return:.2f}%')

print('\n' + '=' * 50)
print('所有因子回测完成!')

## 5. 回测结果分析

In [ ]:
# 生成回测汇总报告
analyzer = BacktestResultAnalyzer(all_results)
summary = analyzer.generate_summary_report(factor_returns)

print('=' * 70)
print('因子偏离策略绩效汇总')
print('=' * 70)
summary_display = summary.copy()
summary_display['总收益率'] = summary_display['总收益率'].apply(lambda x: f'{x*100:.2f}%')
summary_display['年化收益率'] = summary_display['年化收益率'].apply(lambda x: f'{x*100:.2f}%')
summary_display['年化波动率'] = summary_display['年化波动率'].apply(lambda x: f'{x*100:.2f}%')
summary_display['最大回撤'] = summary_display['最大回撤'].apply(lambda x: f'{x*100:.2f}%')
summary_display['夏普比率'] = summary_display['夏普比率'].apply(lambda x: f'{x:.3f}')
summary_display['信息比率'] = summary_display['信息比率'].apply(lambda x: f'{x:.3f}')
print(summary_display.to_string(index=False))

## 6. 可视化结果

In [ ]:
import matplotlib.pyplot as plt
import os

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 创建输出目录
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('可视化配置完成')

In [ ]:
# 绘制各因子相对净值与因子走势对比
for factor in MACRO_FACTORS:
    if factor not in all_results:
        continue
    
    result = all_results[factor]
    portfolio_values = result['portfolio_values']
    benchmark_values = result['benchmark_values']
    relative_values = portfolio_values / benchmark_values
    
    # 因子累计收益
    factor_cumsum = (1 + factor_returns[factor]).cumprod()
    factor_cumsum = factor_cumsum / factor_cumsum.iloc[0]
    
    common_idx = relative_values.index.intersection(factor_cumsum.index)
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # 上图：相对净值走势
    axes[0].plot(relative_values.index, relative_values.values, 
                 label=f'做多{factor}因子', linewidth=1.5, color='blue')
    axes[0].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_title(f'{factor}因子 - 相对净值走势', fontsize=14)
    axes[0].set_xlabel('日期')
    axes[0].set_ylabel('相对净值')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 下图：相对净值与因子走势对比
    axes[1].plot(common_idx, relative_values.loc[common_idx].values,
                label='相对净值', linewidth=1.5, color='blue')
    axes[1].plot(common_idx, factor_cumsum.loc[common_idx].values,
                label=f'{factor}因子', linewidth=1.5, color='orange', alpha=0.7)
    axes[1].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
    axes[1].set_title(f'相对净值 vs {factor}因子走势对比', fontsize=14)
    axes[1].set_xlabel('日期')
    axes[1].set_ylabel('标准化净值')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    save_path = os.path.join(OUTPUT_DIR, f'{factor}_factor_result.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f'{factor}因子图已保存至: {save_path}')
    
    plt.show()

In [ ]:
# 绘制权重偏离图
for factor in MACRO_FACTORS:
    if factor not in all_results:
        continue
    
    result = all_results[factor]
    weights_history = result.get('weights_history', [])
    
    if not weights_history:
        continue
    
    weights_df = pd.DataFrame([w['weights'] for w in weights_history])
    weights_df.index = [w['date'] for w in weights_history]
    
    mean_weights = weights_df.mean()
    base_weights = np.ones(len(ALL_ASSETS)) / len(ALL_ASSETS)
    weight_deviation = mean_weights - base_weights
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    colors = ['green' if x > 0 else 'red' for x in weight_deviation]
    ax.barh(ALL_ASSETS, weight_deviation, color=colors, alpha=0.7)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xlabel('平均权重偏离')
    ax.set_title(f'做多{factor}因子 - 各资产权重偏离', fontsize=14)
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    
    save_path = os.path.join(OUTPUT_DIR, f'{factor}_weight_deviation.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f'{factor}权重偏离图已保存至: {save_path}')
    
    plt.show()

## 7. 宏观风险分解

In [ ]:
from source.risk_analysis import RiskAnalyzer, risk_attribution_analysis

# 假设等权重组合
n_assets = len(ALL_ASSETS)
weights = np.ones(n_assets) / n_assets

# 计算风险分解
portfolio_risk, asset_risk = risk_attribution_analysis(
    weights, 
    exposure_matrix.values, 
    factor_returns, 
    asset_returns,
    ALL_ASSETS, 
    MACRO_FACTORS
)

In [ ]:
# 绘制组合风险分解饼图
import matplotlib.pyplot as plt

risk_data = portfolio_risk[portfolio_risk['风险贡献'] > 0].copy()
risk_data = risk_data.sort_values('风险贡献', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左图：风险分解条形图
factor_risk = risk_data[risk_data['因子'].isin(MACRO_FACTORS)]
hetero_risk = risk_data[~risk_data['因子'].isin(MACRO_FACTORS)]

factor_names = factor_risk['因子'].tolist()
factor_values = factor_risk['风险贡献'].tolist()

if hetero_risk['风险贡献'].sum() > 0:
    factor_names.append('异质风险')
    factor_values.append(hetero_risk['风险贡献'].sum())

colors = plt.cm.Set3(np.linspace(0, 1, len(factor_names)))
axes[0].barh(factor_names, factor_values, color=colors)
axes[0].set_xlabel('风险贡献')
axes[0].set_title('组合宏观风险分解', fontsize=14)
axes[0].grid(True, alpha=0.3, axis='x')

# 右图：风险分解饼图
axes[1].pie(factor_values, labels=factor_names, autopct='%1.1f%%', 
            colors=colors, startangle=90)
axes[1].set_title('风险贡献占比', fontsize=14)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'risk_decomposition.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'风险分解图已保存至: {save_path}')
plt.show()

## 8. 总结

In [ ]:
print('=' * 70)
print('基于宏观因子的大类资产配置框架 - 复现总结')
print('=' * 70)

print('\n【研报核心内容】')
print('1. 六大宏观因子: 增长、通胀、利率、信用、汇率、流动性')
print('2. 四步配置框架:')
print('   - 选取合适的因子（PCA降维 + 资产组合构造）')
print('   - 计算资产的因子暴露（LASSO回归 + 先验信息）')
print('   - 确定因子目标暴露（基准 + 偏离）')
print('   - 匹配因子目标暴露（Blyth-Greenberg优化框架）')

print('\n【复现结果】')
print(f'- 资产数量: {len(ALL_ASSETS)}')
print(f'- 因子数量: {len(MACRO_FACTORS)}')
print(f'- 回测期间: {start_date} 至 {end_date}')
print(f'- 调仓频率: 月度')

print('\n【因子偏离策略表现】')
for _, row in summary.iterrows():
    print(f"- {row['因子']}: 总收益={row['总收益率']*100:.2f}%, 夏普比率={row['夏普比率']:.3f}")

print('\n【输出文件】')
print(f'- 结果保存在: {OUTPUT_DIR}')
print('=' * 70)